# CryFlag — MVP Notebook
## Neonatal Cry Acoustics: Acoustic Screening for Neonatal Stress

**Course:** Applied Machine Learning in Medicine (B.Sc.) — Afeka College  
**Stack:** Python · NumPy · pandas · scikit-learn · XGBoost  
**Fixed seed:** 42 — all operations are deterministic and reproducible  

---

This notebook runs **end-to-end**: preprocessed MFCC features → two-model comparison → 3-state clinical flag.  
The goal is not accuracy maximisation — it is **demonstrated clinical and product reasoning**.  
> **Note:** This system *screens*, *flags*, and *supports* nurse review. It is not a standalone clinical decision-maker.

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")

import os, sys
from pathlib import Path
os.environ.setdefault("MPLCONFIGDIR", str(Path(".matplotlib-cache").resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display
from sklearn.metrics import classification_report

from models import (
    load_data, build_xgboost, build_mlp, evaluate_model,
    plot_confusion_matrix, plot_metric_comparison, FIGURES_DIR, SEED
)
from classify import LABEL_TO_RISK, STATE_NAMES, classify_batch

FIGURES_DIR.mkdir(exist_ok=True)
print(f"Python {sys.version[:6]} | NumPy {np.__version__} | pandas {pd.__version__}")
print("All imports OK.")

---
## Stage 1 — Product Definition

| Field | Value |
|---|---|
| **Product name** | CryFlag |
| **Clinical problem** | Acoustic screening of neonatal cries to *support* nurses in identifying potential physiological or neurological stress |
| **Input X** | Short mono WAV recording (~7 s, 8 kHz) of a neonatal cry |
| **Output Y** | 3-state clinical flag: `Normal` / `Borderline–Suspicious` / `High-Risk` |
| **Target user** | Neonatal nurse / GP |
| **Workflow** | Screening |

**User flow:**  
1. Nurse records a short cry clip on the ward tablet.  
2. CryFlag preprocesses the audio and extracts MFCC features.  
3. The XGBoost model assigns a 3-state risk flag.  
4. If `High-Risk`: nurse is prompted to assess the infant immediately.  
5. If `Borderline–Suspicious`: nurse logs the event and monitors.  
6. If `Normal`: no action required; event logged for continuity.  

**Non-functional requirements:** Inference < 2 s on laptop CPU · No raw audio stored after feature extraction · Output always includes uncertainty level (the 3 probabilities, not just a binary flag).

---
## Stage 2 — Data Preprocessing Summary

**Dataset:** Donate-a-Cry Corpus (`gveres/donateacry-corpus`, GitHub) — ODbL license  
**Sample:** 20 WAV files · 5 labels · 4 files/label · 8 kHz mono PCM-16  
**Split:** 10 train + 5 val + 5 test (seed 42, stratified by label)

| Step | Technique | Clinical Justification |
|---|---|---|
| 1 | Load at 8 kHz mono | Preserves real recorded bandwidth; avoids empty high-freq artefacts from upsampling |
| 2 | Pre-emphasis filter (α = 0.97) | Reduces low-freq handling noise; highlights cry harmonics before spectral analysis |
| 3 | STFT spectrogram (n_fft=512, hop=128) | Converts cry to time-frequency map; burst structure becomes visible |
| 4 | 13 MFCC coefficients (mean + std → 26 features) | Compresses spectral envelope to a lightweight tabular vector; CPU-trainable |

> Label caveat: corpus labels are contributor-provided tags, not verified clinical truth.  
> For a production system, expert annotation by neonatologists would be required.

In [ ]:
# Display Stage 2 before/after preprocessing visuals
preprocessing_figures = [
    ("figures/02_before_after_pre_emphasis.png", "Step 2: Pre-emphasis filter"),
    ("figures/03_before_after_stft_spectrogram.png", "Step 3: STFT spectrogram"),
    ("figures/04_before_after_mfcc.png", "Step 4: MFCC extraction"),
]
for fig_path, title in preprocessing_figures:
    if Path(fig_path).exists():
        print(f"--- {title} ---")
        display(Image(filename=fig_path, width=820))
    else:
        print(f"Figure not found: {fig_path} — run preprocess.py first")

---
## Stage 3 — Model & Metrics

### 3-State Clinical Risk Mapping (5 corpus labels → 3 states)

| Corpus label | Clinical state | Rationale |
|---|---|---|
| `belly_pain` | **High-Risk (3)** | Acute visceral pain signal; highest clinical urgency |
| `discomfort` | **Borderline–Suspicious (2)** | Mild distress; warrants monitoring |
| `burping` | **Borderline–Suspicious (2)** | Physiological but may mask discomfort |
| `hungry` | **Normal (1)** | Routine physiological need; no clinical urgency |
| `tired` | **Normal (1)** | Routine state; no clinical urgency |

### Clinical Metrics — Selection & Justification

| Metric | Justification |
|---|---|
| **Sensitivity (Recall) for High-Risk** | A missed belly_pain cry (false negative) is the highest-cost error — delayed treatment for an infant in pain. We prioritise this metric above all others. |
| **Specificity for High-Risk** | Excessive false alarms cause alert fatigue in nursing staff, eroding trust in the system. Specificity must remain acceptable. |
| **Macro-average AUC (OvR)** | Threshold-independent measure of overall discrimination across all 3 states. More stable than accuracy on small, balanced samples; not inflated by any majority class. |

> **Sample size caveat:** With n=5 test samples, all reported metrics are **illustrative only**.  
> A production system would require hundreds of expert-annotated recordings per class.

In [ ]:
# Load feature table — combine train+val (n=15) for fitting; hold test (n=5) for evaluation
X_train, y_train, X_test, y_test, le, train_df, test_df = load_data()

print(f"Training samples (train + val):  {len(X_train)}")
print(f"Test samples:                    {len(X_test)}")
print(f"Feature dimensions:              {X_train.shape[1]}")
print(f"Label order (LabelEncoder):      {list(le.classes_)}")
print()
print("Train+val label distribution:")
print(train_df["label"].value_counts().to_string())
print()
print("Test label distribution:")
print(test_df["label"].value_counts().to_string())

---
### Model A — XGBoost (Primary Model)

**Architecture:** Gradient-boosted decision trees, max_depth=2, n_estimators=50, lr=0.3  

**Pros:**  
- Handles non-linear MFCC feature interactions natively  
- Scale-invariant (no preprocessing needed)  
- Built-in feature importance — supports clinical interpretability (which MFCC drives the flag?)  
- Consistently strong on small tabular datasets  
- Deterministic with fixed seed  

**Cons:**  
- No temporal structure — treats each MFCC statistic as independent  
- Prone to memorisation on n=15 training samples  

**Why primary:** On small tabular data with engineered features, gradient-boosted trees consistently outperform neural nets. The feature importance output supports clinical explainability — a key non-functional requirement for a medical screening product.

In [ ]:
# Train XGBoost
xgb_model = build_xgboost()
xgb_model.fit(X_train, y_train)

xgb_results = evaluate_model(xgb_model, X_test, y_test, list(le.classes_), "XGBoost")

print(f"XGBoost  |  Accuracy: {xgb_results['accuracy']:.2f}  |  Macro AUC: {xgb_results['macro_auc']:.2f}")
print()
print(classification_report(
    y_test, xgb_results["y_pred"],
    target_names=list(le.classes_),
    zero_division=0
))

In [ ]:
# XGBoost feature importance — which MFCCs drive the clinical flagging?
importances = xgb_model.feature_importances_
from models import FEATURE_COLS
feat_df = pd.Series(importances, index=FEATURE_COLS).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 4), dpi=130)
feat_df.head(10).plot(kind="bar", ax=ax, color="#2a9d8f", edgecolor="#1f2933")
ax.set_title("XGBoost Top-10 Feature Importances (MFCC mean/std)", fontweight="bold")
ax.set_ylabel("Importance score")
ax.set_xlabel("Feature")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "xgb_feature_importance.png", bbox_inches="tight")
plt.show()
print("Figure saved: figures/xgb_feature_importance.png")

---
### Model B — MLP Neural Baseline

**Architecture:** 2 hidden layers (32 → 16 units), ReLU, Adam, StandardScaler  

**Note on CNN substitution:** The original proposal described a 1D CNN. Since PyTorch and TensorFlow  
are outside the project stack, `MLPClassifier` is the appropriate substitute.  
On 26-dimensional **summary features** (MFCC mean + std), the CNN's temporal inductive bias over  
the raw time axis is already discarded by the pooling step — MLP and a shallow 1D CNN are  
functionally equivalent for this feature representation.  

**Pros:**  
- Can learn non-linear feature interactions  
- Feature-scale invariant with StandardScaler prepended  

**Cons:**  
- Requires significantly more data to generalise (n=15 is far below practical minimum)  
- Less interpretable than XGBoost — no built-in feature importance  
- Prone to high variance on tiny datasets: results may differ across random seeds

In [ ]:
# Train MLP baseline
mlp_model = build_mlp()
mlp_model.fit(X_train, y_train)

mlp_results = evaluate_model(mlp_model, X_test, y_test, list(le.classes_), "MLP Baseline")

print(f"MLP      |  Accuracy: {mlp_results['accuracy']:.2f}  |  Macro AUC: {mlp_results['macro_auc']:.2f}")
print()
print(classification_report(
    y_test, mlp_results["y_pred"],
    target_names=list(le.classes_),
    zero_division=0
))

---
### Model Comparison & Primary Selection

In [ ]:
# Side-by-side comparison table
all_results = [xgb_results, mlp_results]
comparison = pd.DataFrame([
    {
        "Model": r["model"],
        "Accuracy": round(r["accuracy"], 3),
        "Macro AUC": round(r["macro_auc"], 3) if not np.isnan(r["macro_auc"]) else "n/a",
    }
    for r in all_results
])
print("=== MODEL COMPARISON ===")
display(comparison)

# Bar chart
fig_path = plot_metric_comparison(all_results, FIGURES_DIR)
display(Image(filename=str(fig_path), width=580))

**Primary model selected: XGBoost**

XGBoost achieves equal or better metrics on this small dataset, trains in under 1 second,  
and provides feature importance scores that support clinical explainability — a mandatory  
non-functional requirement for a medical screening product.  

An MLP with n=15 training samples is dominated by variance; its results are unstable  
across seeds. For a clinical screening tool, *interpretability* and *stability* outweigh  
any marginal accuracy gain from a neural architecture.

---
## Stage 3 — 3-State Clinical Output

The XGBoost model produces 5-class softmax probabilities (one per corpus label).  
These are **collapsed** into 3 clinical states by summing the probabilities of labels  
that share the same risk level:

```
P(High-Risk)   = P(belly_pain)
P(Borderline)  = P(discomfort) + P(burping)
P(Normal)      = P(hungry) + P(tired)
```

**Threshold assignment logic (clinically conservative):**

| Condition | Assigned state | Clinical rationale |
|---|---|---|
| P(High-Risk) ≥ 0.30 | **High-Risk** | Low threshold maximises sensitivity for acute pain; false negatives are more costly than false alarms |
| P(Normal) ≥ 0.55 | **Normal** | Require a clear normal signal before clearing; ambiguity defaults to Borderline |
| Otherwise | **Borderline–Suspicious** | Uncertain cases are flagged, not cleared |

> Thresholds are a clinical product decision, not a statistical optimisation.  
> They would be calibrated against a larger annotated dataset before any clinical deployment.

In [ ]:
# Generate 3-state clinical output for the entire test set
results_3state = classify_batch(
    model=xgb_model,
    X=X_test,
    le=le,
    filenames=list(test_df["filename"]),
    true_labels=list(test_df["label"]),
)

print("=== 3-STATE CLINICAL OUTPUT — TEST SET ===")
display(
    results_3state[[
        "filename", "true_label", "true_risk",
        "P(Normal)", "P(Borderline)", "P(High-Risk)", "Clinical Flag"
    ]]
)

# Save to CSV for the report
results_3state.to_csv("data/processed/test_3state_output.csv", index=False)
print("\nSaved: data/processed/test_3state_output.csv")

In [ ]:
# Confusion matrices for both models (5-class)
for results, model_name in [(xgb_results, "XGBoost"), (mlp_results, "MLP Baseline")]:
    fig_path = plot_confusion_matrix(
        y_test, results["y_pred"],
        list(le.classes_), model_name, FIGURES_DIR
    )
    print(f"--- Confusion Matrix: {model_name} ---")
    display(Image(filename=str(fig_path), width=480))

In [ ]:
# 3-state confusion matrix for XGBoost (collapsed labels)
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

true_risk = results_3state["true_risk"].values
pred_risk = results_3state["state_id"].values
state_labels = ["Normal (1)", "Borderline (2)", "High-Risk (3)"]

cm_3state = confusion_matrix(true_risk, pred_risk, labels=[1, 2, 3])
fig, ax = plt.subplots(figsize=(5, 4), dpi=150)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_3state, display_labels=state_labels)
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title("3-State Confusion Matrix — XGBoost (CryFlag)", fontweight="bold", pad=10)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "confusion_3state_xgboost.png", bbox_inches="tight")
plt.show()
print("Saved: figures/confusion_3state_xgboost.png")

---
## Stage 4 — MVP Completion Checklist

| Item | Status |
|---|---|
| End-to-end pipeline (raw features → 3-state output) | Complete |
| Before/After preprocessing visuals (4 figure pairs) | Complete |
| Two models trained and compared | Complete (XGBoost + MLP) |
| Primary model selected with clinical justification | Complete (XGBoost) |
| Clinical metrics chosen and justified | Complete (Sensitivity, Specificity, Macro AUC) |
| 3-state output with threshold logic | Complete (High-Risk threshold = 0.30) |
| Restricted clinical terminology check | Passed |
| CPU-trainable (under 2 minutes on laptop) | Complete (~2 seconds total) |
| Fixed seed (reproducible) | Seed 42 throughout |
| project_development_log.md updated | Complete |

---

**Clinical use statement:**  
CryFlag *assists* neonatal nurses by *screening* cry recordings and *flagging* those that warrant  
closer clinical attention. It does not replace clinical judgement.  
It must not be used as a standalone clinical decision tool without validation on a larger,  
expert-annotated dataset under appropriate regulatory oversight.